# NBA Player Salaries vs. Performance (2024–25)

**Group Python data project | Web data collection, cleaning, merging, regression, and visualization**

## Project overview

This project investigates whether higher-paid NBA players tend to have stronger on-court performance, measured by **Value Over Replacement Player (VORP)**. I collect player salary rankings and season-level advanced statistics, standardize player names, combine the two datasets, and examine the association using a scatterplot and a simple linear regression.

**Research question:** Among the 100 highest-paid players in the salary dataset, how strongly is annual salary associated with 2024–25 VORP?

**Caution:** This is descriptive analysis, not evidence that performance causes salary changes. Salary reflects contract timing, market factors, and prior performance. AAV (average annual contract value) is not necessarily the player's actual 2024–25 cash salary.

### Sources and reproducibility
- Salary rankings: [Spotrac, NBA player rankings (2025)](https://www.spotrac.com/nba/rankings/player/_/year/2025/sort/cap_total) — source originally used in my project, labeled **AAV Salary** in its extraction.
- Performance: [Basketball-Reference, 2024–25 NBA Advanced Stats](https://www.basketball-reference.com/leagues/NBA_2025_advanced.html).


## 1. Setup

Run cells **from top to bottom** in Colab. The analysis uses `requests`, `beautifulsoup4`, `pandas`, `numpy`, `matplotlib`, `scipy`, and `lxml` (for parsing HTML tables).

In [5]:
%pip -q install beautifulsoup4 lxml scipy openpyxl

In [6]:
import re
import time
import unicodedata
from io import StringIO

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display
import matplotlib.pyplot as plt
from scipy.stats import linregress

SPOTRAC_URL = "https://www.spotrac.com/nba/rankings/player/_/year/2025/sort/cap_total"
BBREF_URL = "https://www.basketball-reference.com/leagues/NBA_2025_advanced.html"
HEADERS = {"User-Agent": "Academic portfolio project (data analysis; contact site for bulk access)"}

# Set to False to skip downloading optional player headshots.
INCLUDE_HEADSHOTS = False

## 2. Collect salary rankings

My original project extracted player names, teams, positions and salary strings from Spotrac list items. Here I retain the same approach, but validate the returned data and parse salaries strictly so malformed strings aren't treated as real dollar amounts.

If this cell fails because Spotrac changed its HTML or blocks automated requests, don't bypass restrictions; use an authorized CSV export or pause and document the limitation.

In [8]:
def fetch_html(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=25)
        response.raise_for_status()
        return response.text
    except requests.RequestException as exc:
        raise RuntimeError(f"Unable to retrieve {url}. Check source availability or permissions.") from exc


def parse_salary(value):
    """Convert a full USD salary string (e.g., '$12,345,678') to dollars."""
    if not isinstance(value, str):
        return np.nan
    match = re.fullmatch(r"\s*\$?\s*(\d{1,3}(?:,\d{3})+|\d+)\s*", value)
    return float(match.group(1).replace(",", "")) if match else np.nan


def extract_spotrac(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for item in soup.select("ul.list-group li.list-group-item"):
        player_tag = item.find("a", class_="link")
        if player_tag is None:
            continue
        rank_tag = item.find("div", class_="fs-3")
        salary_tag = item.find("span", class_="medium")
        info_tag = item.find("small")
        team_and_pos = [x.strip() for x in info_tag.get_text(" ", strip=True).split(",")] if info_tag else []
        rows.append({
            "Rank": rank_tag.get_text(" ", strip=True) if rank_tag else None,
            "Player": player_tag.get_text(" ", strip=True),
            "Team_salary": team_and_pos[0] if team_and_pos else None,
            "Position": team_and_pos[1] if len(team_and_pos) > 1 else None,
            "AAV Salary": salary_tag.get_text(" ", strip=True) if salary_tag else None,
        })
    if not rows:
        raise ValueError("No player rows found on Spotrac. The website layout or access may have changed.")
    salary = pd.DataFrame(rows)
    salary["AAV Salary Value"] = salary["AAV Salary"].map(parse_salary)
    return salary

salary_df = extract_spotrac(fetch_html(SPOTRAC_URL))
print(f"Scraped {len(salary_df):,} salary rows")
print(f"Rows with unparseable salaries: {salary_df['AAV Salary Value'].isna().sum():,}")
display(salary_df.head(10))

RuntimeError: Unable to retrieve https://www.spotrac.com/nba/rankings/player/_/year/2025/sort/cap_total. Check source availability or permissions.

## 3. Collect 2024–25 advanced statistics

VORP estimates a player's contribution relative to a replacement-level player. I also keep **STL%** (steal percentage) from my original notebook as a secondary variable, although the main analysis uses VORP.

For players on multiple teams, Basketball-Reference may report a combined `2TM`/`3TM` row as well as team-specific rows. The combined row is preferred so each player appears once.

In [ ]:
def extract_advanced_stats(html):
    tables = pd.read_html(StringIO(html))
    candidates = []
    for table in tables:
        table.columns = [str(c[-1]) if isinstance(c, tuple) else str(c) for c in table.columns]
        if {"Player", "VORP"}.issubset(table.columns):
            candidates.append(table)
    if not candidates:
        raise ValueError("Could not locate the Basketball-Reference advanced-stats table.")
    stats = candidates[0].copy()
    stats = stats.loc[stats["Player"].astype(str) != "Player"].copy()

    # Basketball-Reference uses 'Team' or 'Tm' depending on the table version.
    team_col = "Team" if "Team" in stats else "Tm" if "Tm" in stats else None
    if team_col is None:
        raise ValueError("Missing team column in advanced-stats table.")
    stats = stats.rename(columns={team_col: "Team_stats"})
    if "STL%" not in stats:
        stats["STL%"] = np.nan
    stats = stats[["Player", "Team_stats", "VORP", "STL%"]].copy()
    for col in ["VORP", "STL%"]:
        stats[col] = pd.to_numeric(stats[col], errors="coerce")
    return stats

stats_df = extract_advanced_stats(fetch_html(BBREF_URL))
print(f"Scraped {len(stats_df):,} advanced-stat rows (may include multi-team duplicates)")
display(stats_df.head(10))

## 4. Clean and merge the datasets

Player names can differ across sources due to accents, suffixes, whitespace or punctuation. Normalizing names improves matching, though ambiguous names still need manual review. We use an **inner join** for the regression so every analyzed player has a valid salary and VORP. The top 100 are selected from **all valid salary records before matching**, not from matched players only.

In [ ]:
def normalize_name(name):
    text = unicodedata.normalize("NFKD", str(name))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower().replace("’", "'")
    text = re.sub(r"\b(jr\.?|sr\.?|ii|iii|iv)\b", "", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

salary_clean = salary_df.copy()
salary_clean["player_key"] = salary_clean["Player"].map(normalize_name)
salary_clean = salary_clean.dropna(subset=["AAV Salary Value"])
salary_clean = salary_clean[salary_clean["AAV Salary Value"] > 0].copy()
salary_clean = salary_clean.sort_values("AAV Salary Value", ascending=False)
salary_clean = salary_clean.drop_duplicates("player_key", keep="first")
salary_top100 = salary_clean.head(100).copy()

stats_clean = stats_df.copy()
stats_clean["player_key"] = stats_clean["Player"].map(normalize_name)
stats_clean["combined_team"] = stats_clean["Team_stats"].astype(str).str.fullmatch(r"\d+TM").fillna(False)
stats_clean = stats_clean.sort_values(["combined_team", "VORP"], ascending=[False, False], na_position="last")
stats_clean = stats_clean.drop_duplicates("player_key", keep="first").drop(columns="combined_team")

matched = salary_top100.merge(stats_clean.drop(columns="Player"), on="player_key", how="left", validate="one_to_one", indicator=True)
unmatched = matched.loc[matched["_merge"] != "both", ["Player", "AAV Salary"]]
print(f"Valid unique salary rows: {len(salary_clean)}")
print(f"Top salary players selected: {len(salary_top100)}")
print(f"Top salary players without a stats match: {len(unmatched)}")
if not unmatched.empty:
    display(unmatched)

analysis_df = matched.loc[(matched["_merge"] == "both") & matched["VORP"].notna()].drop(columns="_merge").copy()
analysis_df = analysis_df.sort_values("AAV Salary Value", ascending=False).reset_index(drop=True)
print(f"Players with usable salary + VORP values: {len(analysis_df)}")
display(analysis_df[["Player", "Team_salary", "Team_stats", "AAV Salary Value", "VORP", "STL%"]].head(10))
if len(analysis_df) < 3 or analysis_df["VORP"].nunique() < 2:
    raise ValueError("Not enough matched observations for regression. Inspect name matches and source data.")

## 5. Explore the relationship

The scatterplot places player VORP on the horizontal axis and average annual salary on the vertical axis. A fitted line summarizes the **linear association**; it is not a prediction of what an individual player should earn.

In [ ]:
result = linregress(analysis_df["VORP"], analysis_df["AAV Salary Value"])
r_squared = result.rvalue ** 2
print(f"Sample size: {len(analysis_df)}")
print(f"Pearson r: {result.rvalue:.3f}")
print(f"R²: {r_squared:.3f}")
print(f"Slope: ${result.slope:,.0f} per VORP point")
print(f"p-value for nonzero slope: {result.pvalue:.4g}")

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(analysis_df["VORP"], analysis_df["AAV Salary Value"] / 1_000_000, alpha=0.75)
x_line = np.linspace(analysis_df["VORP"].min(), analysis_df["VORP"].max(), 100)
ax.plot(x_line, (result.intercept + result.slope * x_line) / 1_000_000,
        linestyle="--", label=f"Linear fit (R² = {r_squared:.3f})")
ax.set(xlabel="2024–25 VORP", ylabel="Average annual salary (USD millions)",
       title="NBA salary vs. VORP among highest-paid players")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig("nba_salary_vorp_scatter.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. Interpret and evaluate limitations

Run the cell below to produce a **data-driven** summary. Be careful not to call a correlation a causal effect. A single season of VORP and contract AAV capture different time frames. Players without a statistical match are excluded from regression, and HTML-source changes can affect reproducibility.

In [ ]:
strength = ("weak" if abs(result.rvalue) < 0.3 else
            "moderate" if abs(result.rvalue) < 0.7 else "strong")
direction = "positive" if result.rvalue > 0 else "negative" if result.rvalue < 0 else "zero"
print(f"In the matched sample of {len(analysis_df)} players, salary and VORP show "
      f"a {strength} {direction} linear association (r={result.rvalue:.3f}, R²={r_squared:.3f}).")
print("This is an association only; the analysis cannot establish causation.")
print("Important limitations: one season of performance, contract AAV vs actual pay, "
      "possible name-matching errors, and selection of high-salary players.")

## 7. Export portfolio outputs


In [ ]:
analysis_df.to_csv("nba_salary_vorp_matched.csv", index=False)
print("Saved: nba_salary_vorp_matched.csv")
print("Saved: nba_salary_vorp_scatter.png")

## 8. Optional: player headshot visualization

In [ ]:
if INCLUDE_HEADSHOTS:
    %pip -q install nba_api
    from nba_api.stats.static import players
    from matplotlib.offsetbox import AnnotationBbox, OffsetImage
    import matplotlib.image as mpimg
    from io import BytesIO

    fig, ax = plt.subplots(figsize=(14, 9))
    for _, row in analysis_df.iterrows():
        matches = players.find_players_by_full_name(row["Player"])
        image_added = False
        if matches:
            image_url = f"https://cdn.nba.com/headshots/nba/latest/1040x760/{matches[0]['id']}.png"
            try:
                image_response = requests.get(image_url, timeout=6)
                if image_response.ok:
                    headshot = mpimg.imread(BytesIO(image_response.content), format="png")
                    artist = AnnotationBbox(OffsetImage(headshot, zoom=0.06),
                        (row["VORP"], row["AAV Salary Value"] / 1_000_000), frameon=False)
                    ax.add_artist(artist)
                    image_added = True
            except (requests.RequestException, ValueError, OSError):
                pass
        if not image_added:
            ax.scatter(row["VORP"], row["AAV Salary Value"] / 1_000_000, alpha=0.65)
    ax.plot(x_line, (result.intercept + result.slope*x_line)/1_000_000,
            linestyle="--", label=f"Linear fit (R²={r_squared:.3f})")
    ax.set(xlabel="2024–25 VORP", ylabel="AAV salary (USD millions)",
           title="NBA player salary vs. VORP (optional headshots)")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("Skipped optional headshots; set INCLUDE_HEADSHOTS = True in Setup to enable.")